In [2]:
# ruff: noqa

import network_catalog
from functools import partial

import pandas as pd
from obspy.core import Trace, UTCDateTime  # type: ignore
import obspy

from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

from tidalseis.datetime_ops import is_time_between
import tidalseis._obspy_validation as vld
from tidalseis.load.network import (
    read_network_traces,
    link_station_trace_paths,
    flatten_station_traces,
    get_network_stream,
)
from tidalseis.catalog.triggering import run_coincidece_trigger, TriggerDict
from tidalseis.catalog import (
    DEFAULT_TRIGGERING,
    DEFAULT_PREPROCESSING,
    PreprocessingConfig,
    preprocess_trace,
    TriggeringConfig,
)


import matplotlib.pyplot as plt
import cmap
import matplotlib.colors as mcolor

plt.switch_backend("qtagg")

# plt.switch_backend("qtagg")

NETWORK = network_catalog.AMERY_ICE_SHELF

DATE_FMT = "%d%m%YT%H%M%S"
TEST_START = NETWORK["network_start"]
TEST_END = TEST_START + timedelta(days=1)
BASE = Path("D:/seismic_data/amery_ice_shelf/trace_data/")


def trim_trace(trace: Trace, start: UTCDateTime, end: UTCDateTime) -> Trace:
    trimmed = trace.copy()

    stats = vld.validate_stats(trace.stats)
    rel_start = start - stats["starttime"]
    rel_end = end - stats["starttime"]
    print(start, stats["starttime"])
    start_idx = np.argmin(np.abs(trace.times() - rel_start))
    end_idx = np.argmin(np.abs(trace.times() - rel_end))
    trimmed.data = trace.data[start_idx:end_idx]

    return trimmed

In [3]:
triggering_config = TriggeringConfig(
    trigger_type="classicstalta",
    trigger_on_threshold=6,
    trigger_off_threshold=5,
    num_coincident_stations=3,
    long_term_average_length=60,
    short_term_average_length=2
)
stream_dir = Path("D:/seismic_data/amery_ice_shelf/stream_data/")
all_triggers: list[TriggerDict] = []

path_list: list[tuple[datetime, Path]] = []
for f in stream_dir.iterdir():
    start_str, end_str = str(f.stem).split("_")
    path_list.append((datetime.strptime(start_str, DATE_FMT), f))

path_list_sorted = sorted(path_list, key=lambda x:x[0])

for n, (_, f) in enumerate(path_list_sorted):
    if n>=15:
        continue
    network_stream: obspy.Stream = obspy.read(f)
    network_stream.merge(fill_value=0)
    # traces: list[obspy.Trace] = [i for i in network_stream.traces]
    triggers = run_coincidece_trigger(network_stream, triggering_config)
    all_triggers.extend(triggers)

128 Events Detected
90 Events Detected
87 Events Detected
91 Events Detected
200 Events Detected
188 Events Detected
207 Events Detected
148 Events Detected
108 Events Detected
151 Events Detected
144 Events Detected
133 Events Detected
309 Events Detected
132 Events Detected
173 Events Detected


In [4]:
tides = pd.read_csv("D:/seismic_data/amery_ice_shelf/tidal_model.txt", delimiter=" ")
tide_height = tides.z
tide_date = tides.date
tide_time = tides.time
tide_datetime: list[tuple[datetime, float]] = []
FMT = "%m-%d-%Y%H:%M:%S"
for i, j, z in zip(tide_date, tide_time, tide_height):
    tide_datetime.append((datetime.strptime(i+j, FMT), float(z)))

In [17]:
base_time = all_triggers[0]["time"]
last_time = all_triggers[-1]["time"]
nbins = float(last_time-base_time) // (3600)
event_timing: list[float] = [float(i["time"] - base_time) for i in all_triggers]
clipped_datetime = np.array([(i-base_time.datetime).total_seconds() for i, j in tide_datetime if i>base_time.datetime and i<last_time.datetime])
clipped_tide_height = np.array([j for i, j in tide_datetime if i>base_time.datetime and i<last_time.datetime])

tide_rate = np.gradient(clipped_tide_height, 3600)

# hist = np.histogram(event_timing, bins=nbins)
f, ax = plt.subplots()
ax2 = ax.twinx()
ax3 = ax.twinx()
_= ax.hist(event_timing, int(nbins), color='k')
ax2.plot(clipped_datetime, clipped_tide_height, color='r', label="Tide Height")
ax3.plot(clipped_datetime, -tide_rate, color='b', label="Tide Rate")
ax2.set_yticks([])
ax3.set_yticks([])
ax.set_ylim(0, 20)
ax2.legend(loc="upper right")
ax3.legend(loc="lower right")

# f, ax = plt.subplots()
# ax2 = ax.twinx()
# ax3 = ax.twinx()
# _= ax.hist(event_timing, int(nbins), color='k')
# ax2.plot(clipped_datetime, tide_rate, color='r')
# ax.set_ylim(0, 20)
# ax2.set_yticks([])
# ax3.set_yticks([])
# # plt.xticks(ticks=np.arange(0, 360), labels=clipped_datetime)
plt.show()

In [ ]:
station_traces = read_network_traces(BASE)
station_traces_linked = link_station_trace_paths(station_traces, BASE)
all_traces = flatten_station_traces(station_traces_linked)
stations_only = [i[0] for i in all_traces]
traces_only = [i[1] for i in all_traces]

preprocessing_config = PreprocessingConfig(
    detrend=True,
    filter_type="Bandpass",
    low_frequency_cutoff=5,
    high_frequency_cutoff=20
)

network_start = np.datetime64(NETWORK["network_start"])
network_end = np.datetime64(NETWORK["network_end"])
oneday = np.timedelta64(timedelta(days=1))

start_range = np.arange(network_start, network_end, step=oneday, dtype="datetime64[s]")
end_range = np.arange(network_start + oneday, network_end+oneday, step=oneday, dtype="datetime64[s]")

# print(start_range)
READABLE_FMT = "%B %d %Y"
all_times = [i[1] for i in all_traces]
active_plot = np.empty((len(start_range), 2))
for n, (st, en) in enumerate(zip(start_range, end_range)):
    check_range_for_traces = partial(is_time_between, begin_time=st, end_time=en)
    all_check = np.array([check_range_for_traces(check_time=i.start) for i in all_times])
    nactive = np.count_nonzero(all_check)
    active_plot[n, :] = (st, nactive)
    start_print = st.item().strftime(READABLE_FMT)
    end_print = en.item().strftime(READABLE_FMT)
    if nactive == 0:
        print(f"({(n+1)/len(start_range):.1%}) // {start_print} --> {end_print} No Traces.")
        continue
    
    print(f"({(n+1)/len(start_range):.1%}) // {start_print} --> {end_print}", end=" ")
    _, network_stream, flag = get_network_stream(
        traces_only, st, en, preprocessing=preprocessing_config
    )

    
    start_filename = st.item().strftime(DATE_FMT)
    end_filename = en.item().strftime(DATE_FMT)
    network_stream.write(f"D:/seismic_data/amery_ice_shelf/stream_data/{start_filename}_{end_filename}.mseed")

In [ ]:
event_list: list[tuple[datetime, list[np.ndarray]]] = []
clr_map = cmap.Colormap("crameri:hawaii").to_mpl()

subset_triggers = []
rng = np.random.default_rng()
ntot = min(10, len(triggers))
for i in rng.choice(np.arange(0, len(triggers)), ntot, replace=False):
    subset_triggers.append((i, triggers[i]))

for event, trig_dict in subset_triggers:
    f, ax = plt.subplots()

    event_data: list[np.ndarray] = []
    norm = mcolor.Normalize(0, len(trig_dict["trace_ids"]))
    for n, id in enumerate(trig_dict["trace_ids"]):
        trace = trace_dict[id]
        trig_start = trig_dict["time"]
        trig_end = trig_dict["time"] + trig_dict["duration"]
        trig_trace = trace.slice(starttime=trig_start-1, endtime=trig_end)
        trig_trace.normalize()
        ax.plot(trig_trace.times(), trig_trace.data + n, color=clr_map(norm(n)))
        ax.axvline(1, color='r', linestyle='--')
        event_data.append(np.stack((trig_trace.times(),trig_trace.data), axis=-1))
    
    event_time: datetime = trig_dict["time"].datetime
    event_list.append((event_time, event_data))
    ax.set_title(f"Event: {event+1} @ {event_time.strftime("%B %d, %Y // %I:%M:%S%p")}")
    event+=1

# Add stations to metadata
print(len(event_list))
plt.show()